---
# `Text Splitters`
---

Context length - No. of tokens which your LLM Models can take as an input in one go

Context-window --> context - input


Text Splitters
- they divide large documents into smaller chunks of Text
- these chunks can be used for embeddings, semantic search, RAG
- Instead of searchin in whole, we can only search in the required chunks - Right piece of Information


LLMs - context length
- GPT 3.5: 16k tokens
- GPT 4.0: 128k tokens

So if a Document is too long, 
- LLMs can't process it at once
- Retreivel becomes inefficient
- Important context might be lost

Text Splitters
- also helps us into Downstream tasks
- Embeddings: Short chunks yield more accurate vectors , helps in converting Text into Numbers So Machine cna understand it
- If our llm is retrieving iinformation from the whole embeddings or whole data embeddings, then Retrievel won't be that much efficient, But on the other hand, Semantic search is happending on a chunk of embeddings, This provides better respose from the LLM Model
- Semantic Search - 

Chunking: Helps us get Rid of hallucination as well
- for Optimizing computation resources - working with smaller chunks, of text can be more memory-efficient and allow for better parallelization of processing task

Types of Text splitters
1. length Bases
2. Text Strcuture Based
3. Document Structure Based
4. Semantic Meaning Based

# `Detailed Notes`

# Text Splitters in LangChain

> **Text Splitter = A component in LangChain that divides large documents into smaller chunks so they can be efficiently embedded, retrieved, and sent to an LLM.**

Text splitting is a **very important part of RAG**.

Think of it simply:

```text
Large Document
      ↓
  Text Splitter
      ↓
Small Chunks
      ↓
 Embeddings
      ↓
 Vector Database
```

---

# 1. Why Do We Need Text Splitters?

Suppose you have a 500-page PDF:

```text
company_handbook.pdf
        ↓
       500 pages
```

You don't want to treat the entire PDF as one giant piece of text.

Instead:

```text
500-page PDF
      ↓
PDF Loader
      ↓
Documents
      ↓
Text Splitter
      ↓
┌──────────────┐
│ Chunk 1      │
│ Chunk 2      │
│ Chunk 3      │
│ Chunk 4      │
│ ...          │
│ Chunk 5000   │
└──────────────┘
```

These smaller pieces can then be embedded and stored in a vector database.

---

# 2. Where Text Splitters Fit in RAG

The complete RAG pipeline is:

```text
                 DATA INGESTION
                      │
                      ↓
                Data Sources
                      ↓
                Document Loader
                      ↓
                  Documents
                      ↓
                Text Splitter
                      ↓
                    Chunks
                      ↓
               Embedding Model
                      ↓
                Vector Store
                      │
              ────────┼────────
                      │
                      ↓
                  User Query
                      ↓
                  Retriever
                      ↓
              Relevant Chunks
                      ↓
                     LLM
                      ↓
                   Answer
```

### Important Interview Point

> **Text splitting happens during the document ingestion/indexing phase, before embeddings are created.**

---

# 3. What is a Chunk?

A **chunk** is a smaller piece of a larger document.

Suppose the original document contains:

```text
LangChain is a framework for building LLM applications.
It provides components for models, prompts, chains,
agents, tools, retrieval and memory.
```

After splitting:

```text
Chunk 1:
LangChain is a framework for building LLM applications.

Chunk 2:
It provides components for models, prompts, chains,
agents, tools, retrieval and memory.
```

The exact chunks depend on the splitter and its configuration.

---

# 4. Why Not Just Send the Entire Document?

There are several reasons.

### 1. Context Window

LLMs have context limits.

A huge document may exceed the model's context window.

### 2. Retrieval Quality

RAG works by finding **relevant pieces** of information.

If your chunk contains an entire 500-page document, retrieval becomes less precise.

### 3. Embedding Quality

Embeddings work better when the text represents a reasonably focused piece of information.

### 4. Cost

Sending unnecessary text to an LLM increases token usage and cost.

### 5. Latency

Smaller retrieved contexts generally reduce unnecessary processing.

---

# 5. Simple Example

Imagine:

```text
Document = 10,000 characters
```

You choose:

```text
chunk_size = 1,000
```

Conceptually:

```text
10,000 characters
        ↓
┌──────────────┐
│ Chunk 1 1000 │
│ Chunk 2 1000 │
│ Chunk 3 1000 │
│ ...          │
│ Chunk 10     │
└──────────────┘
```

Now each chunk can be embedded separately.

---

# 6. Basic LangChain Example

A commonly used splitter is:

`RecursiveCharacterTextSplitter`

Example:

```python
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = splitter.split_documents(documents)
```

Here:

```text
chunk_size = 1000
chunk_overlap = 200
```

---

# 7. Understanding `chunk_size`

`chunk_size` controls approximately how large each chunk should be.

For example:

```python
chunk_size=500
```

means we want chunks around that size.

```text
Document
    ↓
Splitter
    ↓
~500-size chunks
```

Larger:

```python
chunk_size=2000
```

creates larger chunks.

Smaller:

```python
chunk_size=300
```

creates smaller chunks.

---

# 8. What Happens If Chunk Size Is Too Small?

Suppose:

```text
chunk_size = 100
```

You may get:

```text
"LangChain is a framework..."

"for building LLM applications..."

"It provides components..."
```

The problem:

> Important context may be separated across chunks.

Example:

```text
Chunk 1:
"The Transformer architecture uses..."

Chunk 2:
"self-attention to process..."
```

The second chunk alone may lose important context.

---

# 9. What Happens If Chunk Size Is Too Large?

Suppose:

```text
chunk_size = 10,000
```

You may have:

```text
Huge Chunk
     ↓
Lots of unrelated information
```

Problems:

* Less precise retrieval
* More irrelevant context
* Higher token usage
* More noise for the LLM

So we need a balance.

---

# 10. What is `chunk_overlap`?

This is one of the most important concepts.

Suppose:

```text
chunk_size = 1000
chunk_overlap = 200
```

Then chunks overlap.

Conceptually:

```text
             1000 characters
┌───────────────────────────────┐
│          Chunk 1              │
└───────────────────────────────┘
                    ┌───────────────────────────────┐
                    │          Chunk 2              │
                    └───────────────────────────────┘
                    ↑
                 200 overlap
```

So:

```text
Chunk 1
[AAAAAAAAAAAAAAAAAAAAAAAAAAAA]

Chunk 2
                    [AAAAAAAAAAAAAAAAAAAAAAAAAAAA]
```

The overlapping region preserves context between chunks.

---

# 11. Why Do We Need Chunk Overlap?

Consider this sentence:

```text
The company provides health insurance to all
full-time employees after completing three months
of employment.
```

Imagine we split exactly in the middle:

```text
Chunk 1:
The company provides health insurance to all full-time

Chunk 2:
employees after completing three months of employment.
```

The meaning is split.

With overlap:

```text
Chunk 1:
The company provides health insurance to all full-time
employees after completing three

Chunk 2:
employees after completing three months of employment.
```

Now the important relationship appears in both chunks.

### Simple Memory Trick

> **Chunk size controls how much text goes into a chunk.**

> **Chunk overlap preserves context between neighboring chunks.**

---

# 12. `RecursiveCharacterTextSplitter`

This is one of the most commonly used splitters for general-purpose RAG.

```python
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
```

Why is it called **recursive**?

It tries to split text using increasingly smaller separators rather than blindly cutting at arbitrary positions.

Conceptually:

```text
Paragraph
   ↓
Sentence
   ↓
Word
   ↓
Character
```

The goal is to keep meaningful pieces together when possible.

---

# 13. How Recursive Splitting Works

Conceptually, the splitter tries separators such as:

```text
1. Paragraph boundaries
2. Line breaks
3. Spaces
4. Characters
```

So instead of:

```text
"LangChain is a framework...|arbitrary cut..."
```

it tries to preserve natural boundaries.

This usually gives better chunks than blindly slicing every N characters.

---

# 14. Example

Suppose:

```text
# LangChain

LangChain is a framework for developing applications
powered by language models.

It provides components for prompts, models, chains,
agents, tools and retrieval.

LangChain is commonly used to build RAG applications.
```

A recursive splitter tries to preserve paragraph boundaries where possible.

Possible output:

```text
Chunk 1:
LangChain is a framework for developing applications
powered by language models.

Chunk 2:
It provides components for prompts, models, chains,
agents, tools and retrieval.

Chunk 3:
LangChain is commonly used to build RAG applications.
```

---

# 15. Splitting Documents vs Splitting Strings

LangChain splitters can work with different forms of input.

### Split text

```python
chunks = splitter.split_text(text)
```

Returns strings.

```text
[
    "Chunk 1",
    "Chunk 2",
    "Chunk 3"
]
```

### Split documents

```python
chunks = splitter.split_documents(documents)
```

Returns `Document` objects.

```text
Document(
    page_content="Chunk 1",
    metadata={...}
)
```

### Why `split_documents()` is useful for RAG

It preserves document metadata.

For example:

```text
Original:
metadata = {
    source: "rag.pdf",
    page: 10
}
```

After splitting, the chunks can retain relevant metadata.

---

# 16. Important Types of Text Splitters

LangChain provides different strategies because different data needs different splitting methods.

Common categories include:

### 1. Character-based Splitters

Split according to character length.

### 2. Recursive Character Splitter

Try meaningful separators recursively.

### 3. Token-based Splitters

Split based on tokens rather than raw characters.

### 4. Markdown Splitters

Respect Markdown structure.

### 5. HTML Splitters

Respect HTML structure.

### 6. Code Splitters

Split source code according to programming-language structure.

### 7. Semantic Chunking

Split based more on semantic similarity/meaning rather than simple character boundaries.

---

# 17. Character Text Splitter

A simple character-based approach:

```python
from langchain_text_splitters import CharacterTextSplitter

splitter = CharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
```

It is straightforward, but it may not preserve semantic structure as well as recursive or structure-aware approaches.

---

# 18. Token-Based Splitting

LLMs process tokens rather than characters.

A token splitter can split text according to token counts.

Conceptually:

```text
Text
 ↓
Tokenizer
 ↓
Tokens
 ↓
Token Chunks
```

This can be useful when you need tighter control over the model's token-based context limits.

---

# 19. Markdown Text Splitter

Suppose your document is:

```markdown
# LangChain

## Models

Models are used to generate responses.

## Chains

Chains connect multiple operations.

## Agents

Agents can choose tools dynamically.
```

A Markdown-aware splitter can use headings and structure.

Instead of blindly doing:

```text
1000 characters
```

it can preserve:

```text
# LangChain
## Models
## Chains
## Agents
```

This can improve retrieval for structured documentation.

---

# 20. Code Text Splitter

Suppose you have:

```python
def calculate_salary():
    ...
```

or:

```javascript
function loginUser() {
    ...
}
```

A generic character splitter may break code in awkward places.

A code-aware splitter can try to preserve:

```text
functions
classes
methods
blocks
```

This is particularly useful when building:

> **Chat with your GitHub repository**

---

# 21. Semantic Chunking

Traditional splitting:

```text
Every 1000 characters
```

Semantic splitting:

```text
Similar meaning
      ↓
Same chunk
```

For example:

```text
Topic A:
Machine learning is...

Topic A:
Supervised learning uses...

Topic A:
Classification predicts...
```

These may remain together.

Then:

```text
Topic B:
Transformers use attention...
```

could start another chunk.

### Important

Semantic chunking can improve retrieval for some datasets, but it can also be more computationally expensive than simple character-based splitting.

---

# 22. How to Choose a Text Splitter

A useful rule:

| Data                      | Good Starting Strategy           |
| ------------------------- | -------------------------------- |
| Plain text                | `RecursiveCharacterTextSplitter` |
| General RAG               | Recursive splitting              |
| Markdown                  | Markdown-aware splitter          |
| HTML                      | HTML-aware splitter              |
| Source code               | Language/code-aware splitter     |
| Token-sensitive workflows | Token-based splitter             |
| Highly semantic documents | Consider semantic chunking       |

Don't automatically use one splitter for every dataset.

---

# 23. Text Splitting for PDFs

Suppose:

```text
company.pdf
```

Pipeline:

```text
PDF
 ↓
PyPDFLoader
 ↓
Documents
 ↓
RecursiveCharacterTextSplitter
 ↓
Chunks
```

Code:

```python
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = PyPDFLoader("company.pdf")

documents = loader.load()

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = splitter.split_documents(documents)
```

---

# 24. Text Splitting for Thousands of Documents

Suppose:

```text
10,000 PDFs
```

Pipeline:

```text
10,000 PDFs
     ↓
Directory/File Discovery
     ↓
PDF Loaders
     ↓
Documents
     ↓
Text Splitter
     ↓
Chunks
     ↓
Batch Embeddings
     ↓
Vector Database
```

Don't necessarily do:

```text
10,000 PDFs
      ↓
Load everything
      ↓
Split everything
      ↓
Keep everything in RAM
```

Instead use:

```text
Batch
 ↓
Load
 ↓
Split
 ↓
Embed
 ↓
Store
 ↓
Release memory
 ↓
Next batch
```

---

# 25. Important: Chunking Is Not Just About Size

A common beginner mistake is:

> "I'll use `chunk_size=1000` and I'm done."

Chunking should consider:

### 1. Document type

PDF ≠ code ≠ Markdown ≠ legal document.

### 2. Content structure

Headings, paragraphs, tables, sections.

### 3. Retrieval task

What kind of information will users search for?

### 4. Embedding model

Different embedding models have different characteristics and input constraints.

### 5. LLM context

How much retrieved context can your LLM effectively use?

---

# 26. Chunk Size and Retrieval Quality

Imagine:

```text
Chunk A
Machine learning definition
Regression
Classification
Clustering
```

If the user asks:

> "What is regression?"

Chunk A may work.

But if the chunk is:

```text
Chunk A
Machine learning definition

Chunk B
Regression

Chunk C
Classification
```

retrieval may become more precise, but you may lose surrounding context.

Therefore:

> **Chunking is a trade-off between context and precision.**

---

# 27. Chunking and Retrieval

This is a very important concept.

Suppose:

```text
User:
"What is the leave policy?"
```

Your vector database searches chunks.

Good chunk:

```text
Chunk 42:
Employees are entitled to 15 days of annual leave...
```

Bad chunk:

```text
Chunk 42:
Employees...annual...company...Monday...manager...
```

The more focused chunk is often easier to retrieve accurately.

---

# 28. Chunking and Embeddings

Each chunk usually gets its own embedding.

```text
Chunk 1
   ↓
Embedding 1

Chunk 2
   ↓
Embedding 2

Chunk 3
   ↓
Embedding 3
```

Then:

```text
Embedding 1 → Vector DB
Embedding 2 → Vector DB
Embedding 3 → Vector DB
```

At query time:

```text
User Question
     ↓
Query Embedding
     ↓
Similarity Search
     ↓
Top Relevant Chunks
```

---

# 29. Chunking and Context Window

Suppose the LLM has a large context window.

You might be tempted to create enormous chunks.

But:

> **A larger context window does not automatically mean larger chunks are better.**

Retrieval quality still matters.

The goal is:

```text
Enough context
+
Minimal irrelevant information
```

---

# 30. Practical Project: PDF RAG

Let's connect everything you've learned so far.

## Project: AI PDF Assistant

User uploads:

```text
deep_learning.pdf
```

### Step 1 — Document Loader

```text
PDF
 ↓
PyPDFLoader
 ↓
Documents
```

### Step 2 — Text Splitter

```text
Documents
 ↓
RecursiveCharacterTextSplitter
 ↓
Chunks
```

### Step 3 — Embedding

```text
Chunks
 ↓
Embedding Model
 ↓
Vectors
```

### Step 4 — Vector Store

```text
Vectors
 ↓
Chroma / FAISS / Qdrant / etc.
```

### Step 5 — Retrieval

```text
Question
 ↓
Retriever
 ↓
Relevant Chunks
```

### Step 6 — LLM

```text
Question + Chunks
        ↓
       LLM
        ↓
      Answer
```

---

# 31. Project Architecture

```text
                   AI PDF ASSISTANT

                    PDF Documents
                         │
                         ↓
                   PDF Loader
                         │
                         ↓
                     Documents
                         │
                         ↓
                   Text Splitter
                         │
                         ↓
                      Chunks
                         │
                         ↓
                   Embeddings
                         │
                         ↓
                  Vector Database
                         │
             ────────────┼────────────
                         │
                         ↓
                    User Query
                         │
                         ↓
                     Retriever
                         │
                         ↓
                  Relevant Chunks
                         │
                         ↓
                        LLM
                         │
                         ↓
                      Answer
```

---

# 32. How to Tune `chunk_size`

There is **no universal perfect number**.

You might start with:

```python
chunk_size=500
chunk_overlap=50
```

or:

```python
chunk_size=1000
chunk_overlap=200
```

or:

```python
chunk_size=1500
chunk_overlap=200
```

Then evaluate retrieval quality.

### Important Interview Answer

If someone asks:

> "What chunk size do you use?"

Don't say:

> "1000 is always best."

Instead say:

> **"There is no universally optimal chunk size. I start with a reasonable baseline based on the document type and retrieval task, then evaluate retrieval quality and adjust chunk size and overlap accordingly."**

That's a much better answer.

---

# 33. Common Mistakes

## Mistake 1 — Chunking blindly

```text
Every 500 characters
```

without considering document structure.

---

## Mistake 2 — No overlap

This can cause context to be lost across chunk boundaries.

---

## Mistake 3 — Excessive overlap

Suppose:

```text
chunk_size = 1000
chunk_overlap = 900
```

Now many chunks contain almost the same information.

Problems:

* More storage
* More embeddings
* Higher cost
* More duplicate retrieval

---

## Mistake 4 — Very small chunks

You can lose context.

---

## Mistake 5 — Very large chunks

You can introduce irrelevant information and reduce retrieval precision.

---

# 34. Important Interview Questions

## Beginner

### Q1. What is a Text Splitter?

**Answer:**

A Text Splitter divides large documents into smaller chunks so they can be efficiently embedded, stored, retrieved, and provided as context to an LLM.

---

### Q2. Why do we need text splitting in RAG?

**Answer:**

Large documents may exceed context limits and can produce poor retrieval granularity. Splitting creates smaller, more focused units for embedding and retrieval.

---

### Q3. What is `chunk_size`?

**Answer:**

It controls the approximate size of each generated chunk.

---

### Q4. What is `chunk_overlap`?

**Answer:**

It specifies how much content is shared between consecutive chunks to preserve context across chunk boundaries.

---

# 35. Intermediate Questions

### Q5. What is `RecursiveCharacterTextSplitter`?

**Answer:**

It is a commonly used splitter that recursively tries different separators to create appropriately sized chunks while preserving natural text boundaries where possible.

---

### Q6. What is the difference between `split_text()` and `split_documents()`?

**Answer:**

`split_text()` works with raw strings and returns text chunks, while `split_documents()` works with LangChain `Document` objects and preserves their document metadata.

---

### Q7. How do you choose chunk size?

**Answer:**

It depends on the document structure, retrieval task, embedding model, and LLM context. I start with a baseline and evaluate retrieval quality rather than assuming a fixed universal value.

---

# 36. Scenario-Based Questions

### Q8. Your RAG system retrieves incomplete answers. What might be wrong with chunking?

Possible issues:

```text
Chunk too small
     OR
Overlap too small
     OR
Semantic boundaries broken
```

I would inspect retrieved chunks and tune chunk size, overlap, and splitting strategy.

---

### Q9. Your retrieval returns huge chunks containing lots of irrelevant information. What would you change?

**Answer:**

I would consider reducing chunk size, improving the splitting strategy, and evaluating whether structure-aware or semantic chunking would produce more focused chunks.

---

### Q10. Your documents are Markdown files. Would you always use `RecursiveCharacterTextSplitter`?

**Answer:**

Not necessarily. Since Markdown contains meaningful headings and sections, a Markdown-aware splitter may preserve document structure better.

---

### Q11. Your knowledge base contains source code. Which approach would you consider?

**Answer:**

I would consider a language/code-aware splitter so functions, classes, and other logical code structures are less likely to be broken arbitrarily.

---

# 37. Advanced Interview Question

### Q12. How would you optimize chunking for a production RAG system?

A strong answer:

> **I would first understand the document types and retrieval requirements. Then I would choose a suitable splitting strategy, preserve metadata, establish baseline chunk size and overlap, evaluate retrieval metrics such as relevance/recall, inspect failure cases, and tune chunking based on real queries. For structured documents, I'd prefer structure-aware splitting where appropriate.**

---

# 38. 30-Second Revision

> **Text Splitter breaks large documents into smaller chunks for efficient embedding and retrieval.**

Remember:

```text
Document
   ↓
Text Splitter
   ↓
Chunks
```

### Two Important Parameters

```text
chunk_size
→ How large each chunk is

chunk_overlap
→ How much content overlaps between chunks
```

### Common Splitter

```python
RecursiveCharacterTextSplitter
```

### RAG

```text
Loader
 ↓
Documents
 ↓
Splitter
 ↓
Chunks
 ↓
Embeddings
 ↓
Vector DB
 ↓
Retriever
 ↓
LLM
```

---

# 39. 2-Minute Revision

## Text Splitters

Text Splitters divide large documents into smaller chunks.

### Why?

* Manage context limits
* Improve retrieval precision
* Create embedding-friendly units
* Reduce irrelevant context
* Control token usage

### Basic Example

```python
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = splitter.split_documents(documents)
```

### Important Concepts

```text
chunk_size
    ↓
Chunk length

chunk_overlap
    ↓
Shared content between chunks
```

### Common Splitters

```text
Recursive Character
Character
Token-based
Markdown
HTML
Code
Semantic
```

### Selection

```text
Plain Text      → Recursive
Markdown        → Markdown-aware
HTML            → HTML-aware
Source Code     → Code-aware
Token-sensitive → Token-based
Semantic Data   → Semantic chunking
```

### Production RAG

```text
Documents
    ↓
Choose splitting strategy
    ↓
Create chunks
    ↓
Preserve metadata
    ↓
Generate embeddings
    ↓
Store in Vector DB
    ↓
Evaluate retrieval
    ↓
Tune chunk size/overlap
```

### Final Interview Answer

> **Text Splitters in LangChain divide large documents into smaller chunks that can be independently embedded and retrieved. `chunk_size` controls chunk length, while `chunk_overlap` preserves context between neighboring chunks. `RecursiveCharacterTextSplitter` is a common general-purpose choice because it attempts to split at natural boundaries. In production RAG systems, chunking should be chosen based on document structure and retrieval requirements and validated through retrieval evaluation rather than using a fixed size universally.**
